<a href="https://colab.research.google.com/github/keyboki371/Grcu/blob/E%E7%AD%89%E5%85%AC%E5%8B%99%E5%9C%92%E5%B0%8F%E8%83%BD%E6%89%8B/E%E7%AD%89%E5%85%AC%E5%8B%99%E5%9C%92%E5%B0%8F%E5%B9%AB%E6%89%8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
=============================================================================
專案名稱：Selenium SSO 登入與網頁流程自動化框架 (概念驗證 PoC)
開發者：[您的名字/暱稱]
最後更新：2026/04

【免責聲明與使用條款】
1. 本程式碼僅供「學術研究」、「Python Selenium 自動化技術交流」及「個人作品集展示」之用。
2. 程式內已移除所有特定目標網站之 URL、真實 DOM 元素定位器及第三方 AI 破解模組。
3. 嚴禁將本框架修改並應用於任何未經授權之真實伺服器、政府機關系統或違反第三方服務條款之情境。
4. 若使用者擅自修改原始碼進行非法爬蟲或系統攻擊，一切法律責任由使用者自行承擔，與原作者無關。
=============================================================================
"""

import os
import random
import threading
import sys
import ctypes
import json
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# ==========================================
# ★ 基礎工具：切換網頁框架 (安全版)
# ==========================================
def switch_to_frame_by_name(driver, frame_name):
    try:
        driver.switch_to.frame(frame_name)
        return True
    except:
        try:
            driver.switch_to.default_content()
            driver.switch_to.frame(frame_name)
            return True
        except:
            return False

# ==========================================
# 全域變數
# ==========================================
bot_driver = None
chrome_hwnd = None
is_hidden = False
stop_event = threading.Event()
pause_event = threading.Event()

class PrintRedirector:
    def __init__(self, text_widget):
        self.text_widget = text_widget
    def write(self, text):
        self.text_widget.insert(tk.END, text)
        self.text_widget.see(tk.END)
    def flush(self): pass

def smart_sleep(seconds):
    remaining = seconds
    while remaining > 0:
        if stop_event.is_set(): raise Exception("🛑 收到使用者停止指令！")
        if pause_event.is_set():
            time.sleep(0.5)
            continue
        chunk = min(0.5, remaining)
        if stop_event.wait(chunk): raise Exception("🛑 收到使用者停止指令！")
        remaining -= chunk

# ==========================================
# ★ 模組 1：通用型 SSO 登入模擬器
# 展示能力：處理多重登入途徑 (帳密/實體憑證)、例外處理、等待機制
# ==========================================
def login_sso_template(driver, login_method, account, password, pincode):
    print("\n🔑 啟動 SSO 登入程序 (模擬環境)...")
    wait = WebDriverWait(driver, 15)

    # [無害化] 使用 Dummy URL 作為概念展示
    target_url = "https://example.com/sso_login_simulation"
    print(f"導向測試網址: {target_url}")
    # driver.get(target_url) # 實際執行時須替換為合法授權之測試伺服器

    smart_sleep(2) # 模擬網路延遲

    if login_method == "standard_auth":
        print("👤 執行【標準帳號密碼】授權流程...")
        # 概念代碼：
        # wait.until(EC.presence_of_element_located((By.ID, "username_input"))).send_keys(account)
        # driver.find_element(By.ID, "password_input").send_keys(password)
        # driver.find_element(By.ID, "login_button").click()

    elif login_method == "smart_card_auth":
        print("💳 執行【硬體憑證/自然人憑證】授權流程...")
        print("⏳ 模擬呼叫底層讀卡機元件 (需等待硬體回應)...")
        time.sleep(3)
        # 概念代碼：
        # wait.until(EC.visibility_of_element_located((By.ID, "pinCode_input"))).send_keys(pincode)
        # driver.find_element(By.ID, "cert_login_button").click()
        print("⏳ 模擬等待憑證加密簽章 (約 15 秒)...")
        smart_sleep(3) # 縮短模擬時間

    print("⏳ 等待伺服器核發 JWT/Session 通行證...")
    smart_sleep(2)
    print("✅ 成功獲取虛擬授權！")
    return "dummy_window_handle"

# ==========================================
# ★ 模組 2：自動化表單處理引擎 (問卷/測驗)
# 展示能力：DOM 元素尋訪、隨機互動模擬、框架切換
# ==========================================
def auto_form_filler_template(driver, task_name=""):
    print(f"\n📝 啟動自動填表模組 (目標: {task_name})...")
    try:
        # [無害化] 拔除 AI 輔助與特定題庫連線，改為純邏輯概念展示
        print("🤖 掃描目標表單 DOM 結構...")
        smart_sleep(2)

        print("👉 模擬自動勾選邏輯...")
        # 概念代碼：
        # inputs = driver.find_elements(By.XPATH, "//input[@type='radio' or @type='checkbox']")
        # for opt in random.sample(inputs, 3):
        #     driver.execute_script("arguments[0].click();", opt)

        smart_sleep(2)
        print("✅ 表單模擬提交完成！")
        return True
    except Exception as e:
        print(f"⚠️ 表單處理模組異常: {e}")
        return False

# ==========================================
# ★ 核心排程器：自動化任務迴圈
# 展示能力：背景執行、無頭模式設定、任務佇列管理
# ==========================================
def run_automation_task(login_method, account, password, pincode, enable_auto_fill):
    global bot_driver, is_hidden
    try:
        print("\n🚀 初始化 Selenium WebDriver 參數...")
        options = webdriver.ChromeOptions()

        # 基礎穩定性參數
        options.add_argument("--autoplay-policy=no-user-gesture-required")
        options.add_argument("--disable-background-timer-throttling")
        options.page_load_strategy = 'eager'
        options.add_argument("--window-size=1280,720")
        options.add_argument("--mute-audio")
        options.add_argument("--disable-blink-features=AutomationControlled")

        # [無害化] 移除所有破壞瀏覽器安全機制的參數 (如 --disable-web-security)

        print("🌐 啟動虛擬瀏覽器 (PoC 模式，不實際開啟網頁)...")
        # driver = webdriver.Chrome(options=options)
        # driver.set_page_load_timeout(30)
        # bot_driver = driver

        # 模擬登入
        login_sso_template(None, login_method, account, password, pincode)

        # 模擬任務迴圈
        task_list = ["資安認知宣導", "職業安全衛生", "個人資料保護"]
        for task in task_list:
            if stop_event.is_set(): break
            print(f"\n🔎 掃描發現待處理任務：【{task}】")
            smart_sleep(3)

            if enable_auto_fill:
                auto_form_filler_template(None, task)
            else:
                print("🛑 自動填表未啟用，跳過本任務。")

        print("\n🏆 排程清單處理完畢！")

    except Exception as e:
        if "停止指令" in str(e): print("\n🛑 任務已由使用者手動終止。")
        else: print(f"🔄 發生異常：{e}")
    finally:
        btn_start.config(state=tk.NORMAL)
        btn_pause.config(state=tk.DISABLED, text="⏸️ 暫停")
        btn_stop.config(state=tk.DISABLED)

# ==========================================
# UI 介面設計 (Tkinter)
# ==========================================
def start_bot_thread():
    method = var_login_method.get()
    acc = entry_account.get().strip()
    pwd = entry_password.get().strip()
    pin = entry_pin.get().strip()
    auto_fill = var_auto_fill.get()

    stop_event.clear()
    pause_event.clear()

    btn_start.config(state=tk.DISABLED)
    btn_pause.config(state=tk.NORMAL, text="⏸️ 暫停")
    btn_stop.config(state=tk.NORMAL)

    threading.Thread(target=run_automation_task, args=(method, acc, pwd, pin, auto_fill), daemon=True).start()

def toggle_pause():
    if pause_event.is_set():
        pause_event.clear()
        btn_pause.config(text="⏸️ 暫停")
        print("\n▶️ [系統] 解除暫停，繼續執行...")
    else:
        pause_event.set()
        btn_pause.config(text="▶️ 繼續")
        print("\n⏸️ [系統] 進入暫停模式...")

def stop_bot():
    stop_event.set()
    pause_event.clear()
    btn_pause.config(state=tk.DISABLED)
    btn_stop.config(state=tk.DISABLED)

def on_login_method_change():
    method = var_login_method.get()
    if method == "standard_auth":
        entry_account.config(state=tk.NORMAL)
        entry_password.config(state=tk.NORMAL)
        entry_pin.config(state=tk.DISABLED)
    else:
        entry_account.config(state=tk.DISABLED)
        entry_password.config(state=tk.DISABLED)
        entry_pin.config(state=tk.NORMAL)

root = tk.Tk()
root.title("Automation Framework PoC")
root.geometry("540x600")
root.configure(padx=15, pady=10)
root.resizable(False, False)

font_title = ("微軟正黑體", 13, "bold")
font_label = ("微軟正黑體", 10)
font_log = ("Consolas", 9)

lbl_title = tk.Label(root, text="⚙️ Web Automation Framework (PoC)", font=font_title, fg="#2c3e50")
lbl_title.pack(pady=(0, 10))

frame_inputs = tk.Frame(root)
frame_inputs.pack(fill=tk.X)

var_login_method = tk.StringVar(value="standard_auth")
frame_radio = tk.Frame(frame_inputs)
frame_radio.grid(row=0, column=0, columnspan=2, sticky="w", pady=(0, 10))

tk.Radiobutton(frame_radio, text="標準帳號授權", variable=var_login_method, value="standard_auth", font=font_label, command=on_login_method_change).pack(side=tk.LEFT, padx=(0, 10))
tk.Radiobutton(frame_radio, text="硬體憑證授權", variable=var_login_method, value="smart_card_auth", font=font_label, command=on_login_method_change).pack(side=tk.LEFT, padx=10)

lbl_account = tk.Label(frame_inputs, text="測試帳號：", font=font_label)
lbl_account.grid(row=2, column=0, sticky="e", pady=3)
entry_account = tk.Entry(frame_inputs, font=font_label, width=32)
entry_account.grid(row=2, column=1, pady=3, padx=2, sticky="w")

lbl_password = tk.Label(frame_inputs, text="測試密碼：", font=font_label)
lbl_password.grid(row=3, column=0, sticky="e", pady=3)
entry_password = tk.Entry(frame_inputs, font=font_label, show="*", width=32)
entry_password.grid(row=3, column=1, pady=3, padx=2, sticky="w")

lbl_pin = tk.Label(frame_inputs, text="模擬 PIN 碼：", font=font_label)
lbl_pin.grid(row=4, column=0, sticky="e", pady=3)
entry_pin = tk.Entry(frame_inputs, font=font_label, show="*", width=32, state=tk.DISABLED)
entry_pin.grid(row=4, column=1, pady=3, padx=2, sticky="w")

frame_opts = tk.Frame(frame_inputs)
frame_opts.grid(row=5, column=0, columnspan=2, pady=10)

var_auto_fill = tk.BooleanVar(value=True)
chk_auto_fill = tk.Checkbutton(frame_opts, text="啟用表單自動處理模組", variable=var_auto_fill, font=font_label, fg="#27ae60")
chk_auto_fill.pack(side=tk.LEFT, padx=10)

frame_buttons = tk.Frame(root)
frame_buttons.pack(pady=10)

btn_start = ttk.Button(frame_buttons, text="▶ 執行展示", command=start_bot_thread, width=12)
btn_start.grid(row=0, column=0, padx=5)

btn_pause = ttk.Button(frame_buttons, text="⏸️ 暫停", command=toggle_pause, width=10, state=tk.DISABLED)
btn_pause.grid(row=0, column=1, padx=5)

btn_stop = ttk.Button(frame_buttons, text="⏹️ 停止", command=stop_bot, width=10, state=tk.DISABLED)
btn_stop.grid(row=0, column=2, padx=5)

txt_log = scrolledtext.ScrolledText(root, wrap=tk.WORD, font=font_log, bg="#1e1e1e", fg="#00ff00", height=15)
txt_log.pack(fill=tk.BOTH, expand=True)

sys.stdout = PrintRedirector(txt_log)
sys.stderr = PrintRedirector(txt_log)

print("✅ 概念驗證框架 (PoC) 已就緒。")
print("⚠️ 請注意：本程式已解除與任何真實伺服器之綁定。")
print("💡 點擊「執行展示」以觀看多執行緒與模擬調度過程。")

root.mainloop()